# Декораторы
Это ... // to do - дописать

## Самый **простейший декоратор**
Принимает функцию и возвращает функцию, ничего с ней не делает

In [ ]:
# Декоратор
def null_dec(func):     # func - это функция, скобочки тут не используем, т.к. мы ее просто передаем, а не вызываем
    return func

# Обычная функция
def simple_hello(name):
    return f'Hello, {name}'


# Декорирование функции, без потери изначальной функции. Изначальную функцию при этом можно спокойно вызывать
complex_hello = null_dec(simple_hello)

print(complex_hello('Nika'))    # Вызов задекорированной функции
print(simple_hello('Ivan'))     # Мы все еще можем вызвать изначальную функцию, с ней ничего не случилось

Hello, Nika
Hello, Ivan


## **Синтаксический сахар** для декораторов - `@decorator_name`

In [ ]:
# Декоратор
def null_dec(func):
    return func

# Создание функции и одновременно ее декорирование
@null_dec
def simple_hello(name):
    return f'Hello, {name}'

# Конструкция @null_dec перед определением функции это синтаксический сахар для конструкции simple_hello = null_dec(simple_hello)
# Можно заметить, что изначальная функция перезаписывается и поэтому теряется.

# Вызов ЗАДЕКОРИРОВАННОЙ функции. Изначальную функцию теперь вызвать не получится
simple_hello('Who')

'Hello, Who'

Более осмысленный декоратор, который что-то делает.   
Переводит результат оригинальной функции в заглавные буквы

In [4]:
def uppercase(func):
    def wrapper():            # определяем функцию (замыкание/обертка) внутри декоратора, которая будет потом возвращена декоратором
        original = func()            # получаем результат изначальной функции (вот тут функция вызывается, поэтому скобочки пишем)
        modified = original.upper()  # переводим в заглавные буквы
        return modified              # возвращаем измененный результат
    return wrapper            # возвращаем функцию, которая переводит результат оригинальной функции в заглавные буквы


@uppercase
def simple_hello():
    return 'Hello'


simple_hello()

'HELLO'

## Применение **нескольких декораторов** к одной функции.   
Декораторы, написанные поочередно, оборачивают функцию `снизу вверх`

In [5]:
# декоратор, оборачивающий результат функции в HTML теги <strong>
def strong(func):    
    def wrapper():
        return f'<strong>{func()}</strong>'
    return wrapper
    
# декоратор, оборачивающий результат функции в HTML теги <em>
def emphasis(func):  
    def wrapper():
        return f'<em>{func()}</em>'
    return wrapper


# Декораторы будут оборачивать функцию СНИЗУ ВВЕРХ
@strong     # 2. потом внешний декоратор
@emphasis   # 1. сначала выполняется внутренний декоратор (самый нижний)
def simple_hello():
    return 'Hello'

# Без синтакстического сахара это выглядело бы так:
# decorated_hello = strong(emphasis(simple_hello))

simple_hello()

'<strong><em>Hello</em></strong>'

## Передача/перебрасывание **переменных** в декорируемую функцию
Общая идея:

In [ ]:
# Простой декоратор, который никак не изменяет поведение функции
def simple_decorator(func):
    def wrapper(*args, **kwargs):       # функция обертка собирает все позиционные (*args) и ключевые (**kwargs) аргументы, сохраняет в соответствующих переменных
        return func(*args, **kwargs)    # распаковывает args и kwargs с помощью операторов * и ** и вызывает оригинальную функцию с этими параметрами
    return wrapper

Пример:

In [6]:
# Более полезный декоратор - выводит информацию об оригинальной функции (название, переданные аргументы)
def info(func):
    def wrapper(*args, **kwargs):
        print('Вызываем функцию:', func.__name__)
        print('Позиционные аргументы:', args)
        print('Ключевые аргументы:', kwargs)

        result = func(*args, **kwargs)

        print('Функция', func.__name__, 'вернула результат:', result)
        return result
    return wrapper


@info
def simple_hello(name):
    return f'Hello, {name}'

print(simple_hello('Nika'))
print()
print(simple_hello(name='Nika'))

Вызываем функцию: simple_hello
Позиционные аргументы: ('Nika',)
Ключевые аргументы: {}
Функция simple_hello вернула результат: Hello, Nika
Hello, Nika

Вызываем функцию: simple_hello
Позиционные аргументы: ()
Ключевые аргументы: {'name': 'Nika'}
Функция simple_hello вернула результат: Hello, Nika
Hello, Nika


## Использование `@functools.wraps` для сохнанения информации об изначальной функции
Важное дополнение!  
При использовании декоратора, **сигнатура** и **метаданные** изначальной функции (например, название и докстрока) **теряются**.  
Чтобы такого не происходило, нужно использовать декоратор `@wraps` из встроенной библиотеки `functools`.

In [8]:
# Изначальная функция:
def greet(name):
    """Возвращает приветствие с вашим именем"""
    return f"Hello, {name}"

# Декоратор без @functools.wraps:
def bad_uppercase(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs).upper()
    return wrapper

# Декоратор с @functools.wraps
from functools import wraps         # добавляем импорт библиотеки functools

def good_uppercase(func):
    @wraps(func)                    # добавляем декоратор @wraps
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs).upper()
    return wrapper


bad_greet = bad_uppercase(greet)        # точно так же можно было бы использовать @bad_uppercase перед определением функции greet()
good_greet = good_uppercase(greet)      # точно так же можно было бы использовать @good_uppercase перед определением функции greet()

print('Изначальная функция:')
print('Имя:\t\t', greet.__name__)
print('Докстрока:\t', greet.__doc__)
print('\nФункция, задекорированная БЕЗ @functools.wraps:')
print('Имя:\t\t', bad_greet.__name__)
print('Докстрока:\t', bad_greet.__doc__)
print('\nФункция, задекорированная с @functools.wraps:')
print('Имя:\t\t', good_greet.__name__)
print('Докстрока:\t', good_greet.__doc__)

Изначальная функция:
Имя:		 greet
Докстрока:	 Возвращает приветствие с вашим именем

Функция, задекорированная БЕЗ @functools.wraps:
Имя:		 wrapper
Докстрока:	 None

Функция, задекорированная с @functools.wraps:
Имя:		 greet
Докстрока:	 Возвращает приветствие с вашим именем
